In [0]:
WITH params AS (
  SELECT
    30 AS p_window_days,
    lower('cwe') AS p_job_keyword,
    CAST('1001' AS STRING) AS p_include_job_id
),
jobs_latest AS (
  SELECT
    workspace_id,
    CAST(job_id AS STRING) AS job_id,
    name AS job_name,
    change_time,
    ROW_NUMBER() OVER (
      PARTITION BY workspace_id, CAST(job_id AS STRING)
      ORDER BY change_time DESC
    ) AS rn
  FROM system.lakeflow.jobs
),
target_jobs AS (
  SELECT
    j.workspace_id,
    j.job_id,
    j.job_name
  FROM jobs_latest j
  CROSS JOIN params p
  WHERE j.rn = 1
    AND (
      lower(COALESCE(j.job_name, '')) LIKE concat('%', p.p_job_keyword, '%')
      OR j.job_id = p.p_include_job_id
    )
),
timeline_scoped AS (
  SELECT
    t.workspace_id,
    CAST(t.job_id AS STRING) AS job_id,
    CAST(t.run_id AS STRING) AS run_id,
    t.run_name,
    t.trigger_type,
    t.run_type,
    t.result_state,
    t.termination_code,
    t.period_start_time,
    t.period_end_time
  FROM system.lakeflow.job_run_timeline t
  INNER JOIN target_jobs j
    ON t.workspace_id = j.workspace_id
   AND CAST(t.job_id AS STRING) = j.job_id
  CROSS JOIN params p
  WHERE t.period_start_time >= timestampadd(DAY, -p.p_window_days, current_timestamp())
),
run_rollup AS (
  SELECT
    workspace_id,
    job_id,
    run_id,
    MAX(run_name) AS run_name,
    MAX(trigger_type) AS trigger_type,
    MAX(run_type) AS run_type,
    MAX(result_state) AS result_state,
    MAX(termination_code) AS termination_code,
    MIN(period_start_time) AS run_start_time_utc,
    MAX(period_end_time) AS run_end_time_utc,
    SUM(unix_timestamp(period_end_time) - unix_timestamp(period_start_time)) AS run_duration_seconds
  FROM timeline_scoped
  GROUP BY workspace_id, job_id, run_id
),
ranked AS (
  SELECT
    r.workspace_id,
    r.job_id,
    j.job_name,
    r.run_id,
    r.run_name,
    r.trigger_type,
    r.run_type,
    r.result_state,
    r.termination_code,
    r.run_start_time_utc,
    r.run_end_time_utc,
    r.run_duration_seconds,
    ROW_NUMBER() OVER (ORDER BY r.run_end_time_utc DESC, r.run_id DESC) AS rn
  FROM run_rollup r
  LEFT JOIN target_jobs j
    ON r.workspace_id = j.workspace_id
   AND r.job_id = j.job_id
  WHERE r.result_state IS NOT NULL
),
top1 AS (
  SELECT *
  FROM ranked
  WHERE rn = 1
)
SELECT
  COALESCE(t.workspace_id, 'N/A') AS workspace_id,
  COALESCE(t.job_id, 'N/A') AS job_id,
  COALESCE(t.job_name, 'N/A') AS job_name,
  COALESCE(t.run_id, 'N/A') AS run_id,
  COALESCE(t.run_name, 'N/A') AS run_name,
  COALESCE(t.latest_result_state, t.result_state, 'NO_DATA') AS latest_result_state,
  t.trigger_type,
  t.run_type,
  t.termination_code,
  t.run_duration_seconds,
  t.run_start_time_utc,
  from_utc_timestamp(t.run_start_time_utc, 'Asia/Seoul') AS run_start_time_kst,
  t.run_end_time_utc AS latest_time_utc,
  from_utc_timestamp(t.run_end_time_utc, 'Asia/Seoul') AS latest_time_kst,
  current_timestamp() AS generated_at_utc,
  from_utc_timestamp(current_timestamp(), 'Asia/Seoul') AS generated_at_kst
FROM (SELECT 1 AS keep_one_row) s
LEFT JOIN (
  SELECT
    workspace_id,
    job_id,
    job_name,
    run_id,
    run_name,
    result_state,
    trigger_type,
    run_type,
    termination_code,
    run_duration_seconds,
    run_start_time_utc,
    run_end_time_utc,
    result_state AS latest_result_state
  FROM top1
) t
  ON TRUE;